# Session 2: two standard recipes for a flood
**A. Real-world problem.** A culvert (a pipe under a road) must carry the flood of a small
stream. Engineers need two numbers: how much of a storm becomes run-off (a *depth* in mm) and
how big the flow peak is (m3/s). This session builds both with two classic formulas and
applies them to a made-up catchment, Mango Creek.

**Plan.** (1) the rational method and its unit factor 3.6, (2) the SCS curve number run-off,
(3) time of concentration Tc, (4) a design problem. Pipe hydraulics is in `openpipeflow` and
retaining structures in `openretainingwall`; here we stop at the flow in the stream.

*Educational model.* Not a design tool.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: the rational method
`Q = C * i * A / 3.6` with Q the peak flow (m3/s), C the run-off coefficient (0 to 1, the share
of rain that runs off), i the rain intensity (mm/h) and A the area (km2).
Why 3.6? 1 mm/h over 1 km2 is 0.001 m * 1 000 000 m2 = 1000 m3 per hour = 1000/3600 = 0.2778
m3/s, and 1/0.2778 = 3.6.

In [ ]:
def rational_peak(C, i, A):
    """Peak flow in m3/s: C (0..1), i in mm/h, A in km2."""
    return C * i * A / 3.6

## Step 2: the SCS curve number method
Soil and land cover are summarised by a *curve number* CN between 0 and 100 (higher = more
run-off: roofs and roads near 98, forest on sandy soil near 40). From CN we get

    S  = 25400 / CN - 254        (mm, the most water the ground can still hold)
    Ia = 0.2 * S                 (mm, rain lost at the start before any run-off)
    Q  = (P - Ia)^2 / (P - Ia + S)   for P > Ia, otherwise 0

P is the storm rain depth (mm) and Q the run-off depth (mm). The 0.2 is the standard
approximation; it is a rule of thumb, not a law.

In [ ]:
def scs_S(CN):
    """Maximum retention S in mm."""
    return 25400.0 / np.asarray(CN, dtype=float) - 254.0


def scs_runoff(P, CN):
    """Run-off depth Q (mm) from storm depth P (mm) and curve number CN."""
    P = np.asarray(P, dtype=float)
    S = scs_S(CN)
    Ia = 0.2 * S
    x = P - Ia
    safe = np.where(x > 0, x, 0.0)
    return np.where(x > 0, safe ** 2 / (safe + S), 0.0)

## Step 3: time of concentration
Tc is the time water needs to travel from the farthest point of the catchment to the outlet.
The Kirpich formula (metric form, approximate, made for small rural catchments):

    Tc (minutes) = 0.0195 * L^0.77 * S^(-0.385)

L = longest flow path in metres, S = average slope (drop divided by length, no unit).
Why does the design storm last Tc? A shorter storm is more intense but the far corner has not
delivered its water yet; a longer storm is gentler (lower i). The peak is best at D = Tc.
We model the contributing area as A * min(1, D / Tc), a simple approximation.

In [ ]:
def kirpich_tc_min(L_m, slope):
    """Approximate time of concentration in minutes."""
    return 0.0195 * L_m ** 0.77 * slope ** (-0.385)


def idf_made_up(t_min):
    """A MADE-UP rain intensity (mm/h) for a storm of t_min minutes. Real curves come from the weather service."""
    return 2000.0 / (np.asarray(t_min, dtype=float) + 15.0)


def peak_for_duration(C, D_min, A, Tc_min):
    """Rational-style peak (m3/s) for a storm of D_min minutes: only the part of A within reach contributes."""
    frac = np.minimum(1.0, np.asarray(D_min, dtype=float) / Tc_min)
    return C * idf_made_up(D_min) * A * frac / 3.6

## Step 4: Mango Creek (made up)
A = 4.5 km2, longest path 3200 m, drop 64 m, rational C = 0.45 (mixed farmland and houses),
CN = 80, a 100 mm storm.

In [ ]:
MANGO = dict(A=4.5, L=3200.0, drop=64.0, C=0.45, CN=80.0, P=100.0)

if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    m = MANGO
    tc = kirpich_tc_min(m["L"], m["drop"] / m["L"])
    i = float(idf_made_up(tc))
    print(f"Tc = {tc:.1f} min, i(Tc) = {i:.1f} mm/h, rational peak = {rational_peak(m['C'], i, m['A']):.1f} m3/s")
    q = float(scs_runoff(m["P"], m["CN"]))
    print(f"S = {float(scs_S(m['CN'])):.1f} mm, Ia = {0.2 * float(scs_S(m['CN'])):.1f} mm, runoff = {q:.2f} mm,",
          f"volume = {q / 1000 * m['A'] * 1e6:,.0f} m3")

    P = np.linspace(0, 200, 400)
    fig, ax = plt.subplots(figsize=(6.4, 4))
    for cn, col in ((60, "tab:green"), (75, "tab:orange"), (90, "tab:red")):
        ax.plot(P, scs_runoff(P, cn), color=col, label=f"CN {cn} (Ia = {0.2 * float(scs_S(cn)):.1f} mm)")
    ax.plot(P, P, "k:", lw=1, label="all rain runs off (Q = P)")
    ax.set_xlabel("storm rain depth P (mm)")
    ax.set_ylabel("run-off depth Q (mm)")
    ax.set_title("SCS curve number: no run-off until Ia, then Q approaches P - Ia - S")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "scs_curves.png", dpi=120)

    D = np.linspace(5, 150, 300)
    fig2, ax2 = plt.subplots(figsize=(6.4, 4))
    ax2.plot(D, peak_for_duration(m["C"], D, m["A"], tc), color="tab:blue")
    ax2.axvline(tc, color="gray", ls="--")
    ax2.annotate(f"Tc = {tc:.0f} min", (tc, 0.5), xytext=(tc + 6, 3))
    ax2.set_xlabel("storm duration D (min)")
    ax2.set_ylabel("peak flow (m3/s)")
    ax2.set_title("Mango Creek: the worst storm lasts as long as Tc")
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "peak_vs_duration.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Use `scs_runoff` to print the run-off of a 60 mm storm for CN 60, 75 and 90 and check the values against the Section C graph.
2. **Stretch.** Write `culvert_flow(C, A, L, drop)` that returns the rational peak using Kirpich and the made-up intensity curve, and test it on Mango Creek.
3. **Explain.** Explain why a storm longer than Tc gives a smaller peak in the rational method even though more rain falls in total.

## Assessment
- **A. Conceptual.** Why does the SCS curve give zero run-off for small storms, and why does Q stay below P for every storm?
- **B. Numerical.** A 60 mm storm falls on Mango Creek with CN = 75. Find the run-off depth and the volume over 4.5 km2.
- **C. Graph and simulation interpretation.** On the run-off graph, at what storm depth does the CN 60 curve leave zero, and the CN 90 curve? What does the difference mean?
- **D. Design and optimisation challenge.** Design: a plan straightens the channel of Mango Creek so that the slope doubles (0.02 to 0.04). Compute the new Tc and rational peak (C = 0.45, same area) and say whether the downstream culvert needs to be bigger. What do you assume?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("unit factor 3.6: C = 1, i = 1 mm/h, A = 1 km2 gives 1000 m3/h = 0.2778 m3/s",
                   abs(L.rational_peak(1.0, 1.0, 1.0) - 1000.0 / 3600.0) < 1e-12 and abs(1.0 / 3.6 - 0.2778) < 1e-4))
    checks.append(("rational peak scales linearly with C, i and A",
                   abs(L.rational_peak(0.5, 20.0, 3.0) - 2 * L.rational_peak(0.25, 20.0, 3.0)) < 1e-12
                   and abs(L.rational_peak(0.5, 40.0, 3.0) - 2 * L.rational_peak(0.5, 20.0, 3.0)) < 1e-12))
    # worked example: P = 100 mm, CN = 80
    S, Ia, Q = float(L.scs_S(80)), 0.2 * float(L.scs_S(80)), float(L.scs_runoff(100.0, 80))
    hand = (100 - 12.7) ** 2 / (100 - 12.7 + 63.5)
    checks.append((f"worked example: CN 80, P 100 mm gives S 63.5, Ia 12.7, Q 50.54 mm (got {S:.1f}, {Ia:.1f}, {Q:.2f})",
                   abs(S - 63.5) < 1e-9 and abs(Ia - 12.7) < 1e-9 and abs(Q - hand) < 1e-9 and abs(Q - 50.54) < 0.005))
    vol = Q / 1000.0 * 4.5e6
    checks.append((f"run-off volume over 4.5 km2 is about 227 thousand m3 (got {vol:,.0f})", abs(vol - 227426) < 5))
    checks.append(("no run-off for P at or below Ia (exact boundary 12.7 mm gives 0)",
                   float(L.scs_runoff(12.7, 80)) == 0.0 and float(L.scs_runoff(5.0, 80)) == 0.0 and float(L.scs_runoff(0.0, 98)) == 0.0))
    checks.append(("just above Ia the run-off is tiny but positive", 0.0 < float(L.scs_runoff(13.0, 80)) < 0.01))
    P = np.linspace(0, 300, 301)
    checks.append(("run-off grows with P (monotonic) and stays below P", bool(np.all(np.diff(L.scs_runoff(P, 80)) >= 0)
                                                                      and np.all(L.scs_runoff(P[1:], 80) < P[1:]))))
    cns = np.arange(40, 99, 2)
    checks.append(("run-off never falls as CN rises (60 mm storm; zero for very low CN)", bool(np.all(np.diff(L.scs_runoff(60.0, cns)) >= 0) and L.scs_runoff(60.0, 98) > L.scs_runoff(60.0, 60))))
    big = 1.0e6
    qb = float(L.scs_runoff(big, 80))
    checks.append(("huge storm: Q / (P - Ia) tends to 1 and the water kept after Ia tends to S = 63.5 mm",
                   abs(qb / (big - 12.7) - 1) < 1e-4 and abs((big - 12.7 - qb) - 63.5) < 0.01))
    checks.append(("CN 100 means S = 0: all rain runs off", float(L.scs_S(100)) == 0.0 and abs(float(L.scs_runoff(60.0, 100)) - 60.0) < 1e-12))
    q3 = L.scs_runoff(60.0, [60, 75, 90])
    checks.append((f"60 mm storm: CN 60, 75, 90 give 3.49, 14.52, 35.78 mm (got {q3[0]:.2f}, {q3[1]:.2f}, {q3[2]:.2f})",
                   np.allclose(q3, [3.49, 14.52, 35.78], atol=0.005)))
    tc = L.kirpich_tc_min(3200.0, 64.0 / 3200.0)
    checks.append((f"Kirpich Tc for Mango Creek is about 44.0 min (got {tc:.1f})", abs(tc - 44.0) < 0.05))
    checks.append(("doubling the slope cuts Tc by 2^-0.385 = 0.766 (about 23 percent)",
                   abs(L.kirpich_tc_min(3200.0, 0.04) / tc - 2 ** -0.385) < 1e-12 and abs(2 ** -0.385 - 0.766) < 5e-4))
    i = float(L.idf_made_up(tc))
    pk = L.rational_peak(0.45, i, 4.5)
    checks.append((f"design: i(Tc) = 33.9 mm/h and rational peak = 19.1 m3/s (got {i:.1f}, {pk:.1f})",
                   abs(i - 33.9) < 0.05 and abs(pk - 19.1) < 0.05))
    D = np.concatenate([np.linspace(5, 150, 2000), [tc]])
    q = L.peak_for_duration(0.45, D, 4.5, tc)
    checks.append(("the peak over all storm durations is largest at D = Tc", abs(D[q.argmax()] - tc) < 1e-9))
    q20, q90 = float(L.peak_for_duration(0.45, 20.0, 4.5, tc)), float(L.peak_for_duration(0.45, 90.0, 4.5, tc))
    checks.append((f"a 20 min storm gives 14.6 m3/s and a 90 min storm 10.7 m3/s, both below the Tc storm (got {q20:.1f}, {q90:.1f})",
                   abs(q20 - 14.6) < 0.05 and abs(q90 - 10.7) < 0.05 and max(q20, q90) < pk))
    tc2 = L.kirpich_tc_min(3200.0, 0.04)
    i2 = float(L.idf_made_up(tc2))
    pk2 = L.rational_peak(0.45, i2, 4.5)
    checks.append((f"straightening the channel (slope doubled): Tc 33.7 min, i 41.1 mm/h, peak 23.1 m3/s, 21 percent more (got {tc2:.1f}, {i2:.1f}, {pk2:.1f})",
                   abs(tc2 - 33.7) < 0.05 and abs(i2 - 41.1) < 0.05 and abs(pk2 - 23.1) < 0.05 and abs(pk2 / pk - 1.21) < 0.005))
    v75 = float(L.scs_runoff(60.0, 75)) * 4.5
    checks.append((f"60 mm on CN 75 over 4.5 km2 gives 65.3 thousand m3 (got {v75:.1f}); Ia is 33.9 mm for CN 60 and 5.6 mm for CN 90",
                   abs(v75 - 65.3) < 0.05 and abs(0.2 * float(L.scs_S(60)) - 33.9) < 0.05 and abs(0.2 * float(L.scs_S(90)) - 5.6) < 0.05))
    checks.append(("100 mm on CN 60 gives 18.6 mm of run-off", abs(float(L.scs_runoff(100.0, 60)) - 18.6) < 0.05))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
